# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` values for clarity and reproducibility.

In [ ]:
# List all record sets, their @id and their fields
for record_set in metadata.record_sets:
    print(f"Record Set Name: {record_set.name}\n  @id: {record_set.id}\n  Description: {record_set.description}")
    print("  Fields:")
    for f in record_set.fields:
        print(f"    - {f.name} (@id: {f.id}, data type: {f.data_type})")
    print('-'*60)

## 3. Data Extraction
Load data from available record sets into pandas DataFrames. Here, we use the `@id` values for referencing record sets and fields. Choose a specific record set you'd like to explore further.

In [ ]:
# Prepare a list of record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\tColumns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"\tFailed to load records: {e}")
    print('-'*30)

# For demonstration, select the first available record set
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id and first_record_set_id in dataframes:
    print(f"\nSample data for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You should select appropriate numeric and grouping fields by their `@id`.

In [ ]:
# EDA on the main record set (choose by @id)
# You may want to change these to specific available field @ids from the overview above
record_set_id = first_record_set_id
df = dataframes.get(record_set_id)

if df is not None:
    # Automatically detect numeric fields
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print("Numeric fields:", numeric_fields)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()  # Use mean as threshold example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}' column (z-score):")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a categorical (string/object) column
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No non-numeric field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Plots below use the filtered and normalized data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    fig, ax = plt.subplots(figsize=(8,4))
    # Histogram of the first numeric field
    sns.histplot(df[numeric_fields[0]], kde=True, ax=ax)
    ax.set_title(f"Histogram of '{numeric_fields[0]}'")
    plt.show()

    if group_field_id is not None:
        # Boxplot of the numeric field grouped by the group field
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_fields[0], data=df)
        plt.title(f"Boxplot of '{numeric_fields[0]}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot visualize data: suitable DataFrame or columns not found.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and process the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. By using entity `@id` references, we ensured reproducibility and clarity. The basic analysis steps included summary exploration of record sets and fields, filtering and normalizing numeric values, and visualizing distributions.

<br/>
You can extend this notebook by:
- Exploring other record sets
- Customizing preprocessing steps based on your analysis goals
- Creating additional visualizations

For full schema details, refer to the Croissant JSON-LD definition linked above.